# 19 — Channel factorial: belief accuracy vs closed-loop profit

One **joint shard** per `(seed, ObsChannels)` runs a single closed-loop `act()` episode and records **mean-f MAE**, **8-bin distribution MAE**, and **profit** from the same scored days.

Canonical grid: `(upc|gsin) × (waste on|off) × (none|pack_date|temperature_history)` — 12 cells.

Budget target: ~20 min wall / 2 CPU-hr on Modal (probe → plan → run).

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd

REPO_ROOT = Path.cwd() if (Path.cwd() / "src" / "blueberries_voi").is_dir() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "experiments" / "data"
FIG_DIR = REPO_ROOT / "figures" / "channel_joint"
OUT_JSON = DATA_DIR / "nb19_joint_rows.json"

_wheel_dir = REPO_ROOT / "dist" / "wheel"
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")
if _wheel_dir.is_dir():
    os.environ["BLUEBERRIES_VOI_WHEEL"] = str(_wheel_dir)

from blueberries_voi.experiments.batch_budget import assert_within_budget, plan_channel_joint_budget
from blueberries_voi.experiments.channel_factorial_viz import save_nb19_figures
from blueberries_voi.experiments.channel_joint import (
    all_obs_channels_product,
    channel_joint_job_grid,
    run_seed_channel_joint,
)
from blueberries_voi.experiments.modal_dispatch import run_batch
from blueberries_voi.filter.types import channels_for_preset

BATCH_MODE: Literal["modal", "local"] = "modal"
SMOKE = False
ACCURACY_METRIC: Literal["mean_f", "distribution"] = "mean_f"

PROBE_SEED = 42
PROBE_CHANNEL = channels_for_preset("P0")
CHANNELS = all_obs_channels_product()
CANDIDATE_SEEDS = (42, 7, 99, 101, 2024, 31415)

## Probe — one shard wall time

In [2]:
probe_t0 = time.perf_counter()
probe_row = run_seed_channel_joint(
    PROBE_SEED,
    PROBE_CHANNEL,
    n_burn=2,
    n_score=10,
)
probe_elapsed_s = time.perf_counter() - probe_t0
print(f"probe elapsed_s={probe_elapsed_s:.1f} mae_f={probe_row['mae_f']:.4f} profit={probe_row['profit']:.2f}")

probe elapsed_s=1.9 mae_f=0.1291 profit=347.00


## Plan — greedy seeds then bump `n_score` under Modal budget

In [3]:
plan = plan_channel_joint_budget(probe_elapsed_s, max_seeds=len(CANDIDATE_SEEDS))
assert_within_budget(plan)
SEEDS = tuple(CANDIDATE_SEEDS[:plan.n_seeds])
N_BURN = plan.n_burn
N_SCORE = plan.n_score
print(plan.as_dict())
print(f"grid={len(SEEDS)} seeds × {len(CHANNELS)} channels = {len(SEEDS) * len(CHANNELS)} shards")

{'n_seeds': 6, 'n_score': 30, 'n_burn': 2, 'n_channels': 12, 'shard_count': 72, 't_shard_s': 1.8803427840030054, 'est_wall_s': 5.641028352009016, 'est_cpu_hr': 0.03760685568006011}
grid=6 seeds × 12 channels = 72 shards


## Run — Modal or local batch

In [4]:
run_t0 = time.perf_counter()
if OUT_JSON.is_file() and not SMOKE:
    rows = json.loads(OUT_JSON.read_text(encoding="utf-8"))
    run_wall_s = 0.0
    print(f"loaded {len(rows)} rows from {OUT_JSON.relative_to(REPO_ROOT)}")
else:
    rows = run_batch(
        "channel_joint",
        BATCH_MODE,
        smoke=SMOKE,
        seeds=SEEDS,
        channels=CHANNELS,
        n_burn=N_BURN,
        n_score=N_SCORE,
        out_path=OUT_JSON,
    )
    run_wall_s = time.perf_counter() - run_t0
    print(f"run wall_s={run_wall_s:.1f} rows={len(rows)}")
df = pd.DataFrame(rows)
df.head()

loaded 72 rows from experiments/data/nb19_joint_rows.json


,seed,key,profit,stockout,code_type,scan_waste,delivery_history,preset,waste_total,waste,delivery,mae_f,mae_dist,n_burn,n_score,n_live_days
0,7,code=gsin|waste=0|hist=none,588.0,135,gsin,False,none,custom,146,off,none,0.126234,0.090409,2,30,29
1,42,code=gsin|waste=0|hist=none,1052.5,69,gsin,False,none,custom,83,off,none,0.094936,0.069339,2,30,30
2,99,code=gsin|waste=0|hist=none,1090.5,35,gsin,False,none,custom,103,off,none,0.083951,0.066119,2,30,30
3,101,code=gsin|waste=0|hist=none,762.0,116,gsin,False,none,custom,120,off,none,0.126899,0.087602,2,30,29
4,2024,code=gsin|waste=0|hist=none,849.5,97,gsin,False,none,custom,121,off,none,0.108326,0.089485,2,30,30


## Audit — shard coverage and CPU estimate

In [5]:
expected = len(channel_joint_job_grid(SEEDS, CHANNELS))
assert len(rows) == expected, (len(rows), expected)
audit_path = DATA_DIR / "nb19_run_audit.json"
if audit_path.is_file():
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    audit["accuracy_metric"] = ACCURACY_METRIC
else:
    est_cpu_hr = (len(rows) * probe_elapsed_s) / 3600.0
    audit = {
        "shards": len(rows),
        "seeds": list(SEEDS),
        "n_score": N_SCORE,
        "n_burn": N_BURN,
        "probe_elapsed_s": probe_elapsed_s,
        "run_wall_s": run_wall_s,
        "est_cpu_hr": est_cpu_hr,
        "accuracy_metric": ACCURACY_METRIC,
    }
print(json.dumps(audit, indent=2))
agg = df.groupby(["code_type", "waste", "delivery"], observed=True).agg(
    mae_f=("mae_f", "mean"),
    mae_dist=("mae_dist", "mean"),
    profit=("profit", "mean"),
)
agg

{
  "shards": 72,
  "seeds": [
    42,
    7,
    99,
    101,
    2024,
    31415
  ],
  "n_score": 30,
  "n_burn": 2,
  "probe_elapsed_s": 1.8803427840030054,
  "run_wall_s": 0.0,
  "est_cpu_hr": 0.03760685568006011,
  "accuracy_metric": "mean_f"
}


mae_f  mae_dist      profit
code_type waste delivery                                           
gsin      off   none                 0.109540  0.081838  819.750000
                pack_date            0.035252  0.040216  888.333333
                temperature_history  0.017022  0.033929  902.083333
          on    none                 0.135112  0.092981  712.000000
                pack_date            0.038752  0.044879  877.666667
                temperature_history  0.025528  0.038545  865.083333
upc       off   none                 0.111804  0.083672  842.666667
                pack_date            0.033101  0.040327  884.166667
                temperature_history  0.017528  0.033566  902.083333
          on    none                 0.113053  0.085969  837.416667
                pack_date            0.036873  0.042499  873.250000
                temperature_history  0.018084  0.033790  890.916667

## Plots

In [6]:
acc_col = "mae_f" if ACCURACY_METRIC == "mean_f" else "mae_dist"
written = save_nb19_figures(rows, FIG_DIR, accuracy_column=acc_col)
for path in written:
    print(path.relative_to(REPO_ROOT))

/home/oliver/blog/blueberries-voi/.worktrees/T-nb19-implement/src/blueberries_voi/experiments/channel_factorial_viz.py:76: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


figures/channel_joint/channel_factorial_heatmap_mae_f.png
figures/channel_joint/profit_vs_mae_f.png
figures/channel_joint/parallel_coords_mae_f.png
